# Polytope Lib Tests

In [1]:
from pycvxset import Polytope
import numpy as np
import matplotlib.pyplot as plt

import os
import mosek

os.environ["MOSEKLM_LICENSE_FILE"] = "27007@192.168.100.118"
mosek.Env().getversion()
mosek.Env().putlicensepath("27007@192.168.100.118")
mosek.Env().expirylicenses()

-1

In [2]:
from itertools import product
def cartesian_product(*polytopes):
    V_rep = [p.V for p in polytopes]
    Combinations = []
    for veritces in product(*V_rep):
        Combinations.append(np.concatenate(veritces))
    return Polytope(V=np.array(Combinations))

In [3]:
# Define the polytope using H-representation with 2D

A2 = np.array([[1, 0], [0, 1], [-1, 0], [0, -1]])
b2 = np.array([1, 2, 1, 2])

P_2D = Polytope(A=A2, b=b2)

# Define the polytope using H-representation with 1D

P_1D = Polytope(A=np.array([[1], [-1]]), b=np.array([3, 3]))

# Define the polytope using H-representation with 3D as Chartesian product of 1D and 2D

P = cartesian_product(P_1D, P_2D)


ax, _, _ = P.plot()
plt.show() 

KeyboardInterrupt: 

In [ ]:
F = Polytope(A = P.A/P.b[:,np.newaxis], b= np.ones((len(P.b),1)))
#ax, _, _ = F.plot()
#plt.show

P.A[:,:1],P.A[:,1:]

In [ ]:
# Check proportional scalebility of sets

sigP = 0.5 * P

ax, _, _ = sigP.plot()
plt.show() 


# Casadi Tests

In [ ]:
import numpy as np
import casadi as ca 
from pycvxset import Polytope

In [ ]:
N = 30
n = 3
m = 2
op = ca.Opti()

x_opti =op.variable(n)
op.set_initial(x_opti,[1,2,3])
print(x_opti)

Q = ca.MX(np.array([[1,0,0],[0,2,0],[0,0,3]]))

# A = np.random(n,n)
# B = np.random(n.m)

x = ca.MX.sym('x',n,1,N)
x[0]=ca.MX(np.array([1,2,3]))
u = ca.MX.sym('u',m,1,N)
print(Q)
print(Q@x[0])
# constraint = []

# for i in range(N):

#     constraint.append(x[i+1]==A @ x[i] + B @ u[i])
#     ca.qpsol()



## Casadi - Solving a NLP with Opti

In [ ]:
opti = ca.Opti()

x = opti.variable()
y = opti.variable()

opti.minimize((y-x**2)**2)
opti.subject_to(x**2+y**2==1)
opti.subject_to(x+y>1)

opti.solver('ipopt')

sol = opti.solve()

print(sol.value(x))
print(sol.value(y))

## Casadi - Solving a NLP with MX

In [ ]:
x = ca.MX.sym('x')
y = ca.MX.sym('y')
z = ca.MX.sym('z')

F = ca.MX.sym("F")
G = ca.MX.sym("G")
f = (y-x**2)**2
g = ca.vertcat(x**2+y**2,x+y)
nlp = {'x':ca.vertcat(x,y), 'f':f, 'g':g, 'p':z}
s = ca.nlpsol('s', 'ipopt', nlp)
r = s(lbg=ca.vertcat(1,1+z), ubg=ca.vertcat(1,np.abs(-np.inf)),x0=0.786151)
x_opt = r['x']
print(x_opt)


In [ ]:
z = ca.vertcat(x,y)
Const = ca.horzcat(F,G)
Const@z
g = ca.vertcat(g,Const@z)
const_value = []
h = np.array([[1,1,1],[2,2,2],[3,3,3]])
H = ca.MX(h)
print(H)

## Transforming a MX object into a numpy array - need to pass by DM object

In [ ]:
print(H)

g=ca.MX()

g=ca.vertcat(g,H)
g

# CVXPY LMI Solver

In [ ]:
import cvxpy as cp
import numpy as np

import os
import mosek

os.environ["MOSEKLM_LICENSE_FILE"] = "27007@192.168.100.118"
mosek.Env().getversion()
mosek.Env().putlicensepath("27007@192.168.100.118")
mosek.Env().expirylicenses()

# Data
A = np.array([[0, 1], [-1, -2]])
n = 2

# Variables
P = cp.Variable((n, n), symmetric=True)

tol = cp.Parameter(1,"tol")

# Constraints
# P > 0 and A.T*P + P*A < 0 (using a small epsilon for strict inequality)
constraints = [P >> tol * np.eye(n), 
               A.T @ P + P @ A << -tol * np.eye(n)]

# Problem
prob = cp.Problem(cp.Minimize(0), constraints)
#prob.solve()

#print("Status:", prob.status)
#print("P:\n", P.value)


In [ ]:
tol.value = [1e-6]
prob.solve()
print("Status:", prob.status)
print("P:\n", P.value)

In [ ]:
import scipy as sc
l=cp.Parameter(1,"lambda")
print(l)
l.shape

A = np.array([[1,2],[2,1]]).transpose()
print(np.linalg.inv(A))

In [ ]:
rho = np.array([[.9, .1, 2]])

if np.isnan(rho).any():
    print("Hello")

np.isnan(rho).any()

In [ ]:
import numpy as np
A=np.array([[[1, 2],[3, 4]],
            [[5, 6],[7, 8]],
            [[9, 10],[11, 12]]])
len(A.shape)
A_trans = np.copy(A).T

B=np.array([[1, 2],[3, 4]])
len(B.shape)

if len(A.shape)>2:
    v=A.shape
else:
    v=1


if len(B.shape)==2:
    B = B[np.newaxis, :, :]

b = np.array([1, 0])

B @ b

# Example of A[0]*a[0] + A[1]*a[1] + A[2]*a[2] ... where a is a vector of parameters and the first is always 1

a = np.array([0, 1, 0],).T
A = A.transpose(1, 2, 0)

A_trans @ a, A @ a, A[:, :, 0]


## pagemtimes() subsitution

In [ ]:
import numpy as np









# Simple MPC QP with CasADi

## 1st attempt 

In [ ]:
import numpy as np
import casadi as ca
import osqp

x0=np.array([[10],[0]])
u0=np.array([[0]])

# System Matricies

A = ca.MX(np.array([[1, 1],[0, 1]]))
B = ca.MX(np.array([[0], [1]]))
C = ca.MX(np.array([[1,0]]))


N = 2 # Prediction horizon 


x = ca.MX.sym('x',2,N+1)
u = ca.MX.sym('u',1,N)
y = ca.MX.sym('y',1,N)
X = ca.MX.sym('X',2)
U = ca.MX.sym('U',1)
lti=ca.Function('lti',[X,U],[A@X+B@U,C@X],['x_0','u_0'],['x_1','y_0'])

f = 0
Q = 1 
R = 0.1
P = np.eye(2)


for i in range(N):
    f +=y[:,i].T@Q@y[:,i]+u[:,i].T@R@u[:,i]

f += x[:,-1].T@P@x[:,-1]

g=ca.vertcat(x[:,0],ca.reshape(ca.vertcat(
              u, 
             (y - C@x[:,1:]),
             (x[:,1]-A@x[:,:-2]-B@u)),-1,1))

lbg = ca.vertcat(lti(x0,u0),-np.ones((N,1)),np.zeros((N,1)),np.zeros((N*2,1)))
ubg = ca.vertcat(lti(x0,u0),np.ones((N,1)),np.zeros((N,1)),np.zeros((N*2,1)))

qp = {'f':f, 'x':ca.vertcat(ca.reshape(x,-1,1),ca.reshape(u,-1,1),ca.reshape(y,-1,1)), 'g': g}

mysol = ca.qpsol('mysol', 
                 'qpoases', 
                 qp)

display(mysol)
mysol(lbg=lbg, ubg=ubg)




## 2nd attempt - iterative contruction of cost and constraints

In [ ]:
import casadi as ca
import numpy as np
# --- System Setup ---
n_x, n_u, n_y = 2, 1, 1
N = 10 # Longer horizon for testing

# Matrices (use DM for numeric constants, MX for symbolic)
A = ca.DM([[1, 1], [0, 1]])
B = ca.DM([[0], [1]])
C = ca.DM([[1, 0]])
Q, R, P = 1, 0.1, np.eye(2)

# --- Symbolic Formulation ---
w = []      # Decision variables (x0, u0, x1, u1...)
lbw, ubw = [], []
g = []      # Constraints (dynamics and output)
lbg, ubg = [], []
J = 0       # Cost function

# Parameter for initial state (to allow MPC in a loop)
x_init = ca.MX.sym('x_init', n_x)

# Current state (starting from symbolic x0)
Xk = ca.MX.sym('X_0', n_x)
w.append(Xk)
lbw.extend([-ca.inf]*n_x); ubw.extend([ca.inf]*n_x)

# Constraint x0 = x_init
g.append(Xk - x_init)
lbg.extend([0]*n_x); ubg.extend([0]*n_x)

for k in range(N):
    Uk = ca.MX.sym(f'U_{k}', n_u)
    w.append(Uk)
    # Input bounds (very simple here)
    lbw.append(-1.0); ubw.append(1.0)
    
    # Output and stage cost
    Yk = C @ Xk
    J += Yk.T @ Q @ Yk + Uk.T @ R @ Uk
    
    # Dynamics (equality constraint)
    X_next = A @ Xk + B @ Uk
    Xk = ca.MX.sym(f'X_{k+1}', n_x)
    w.append(Xk)
    lbw.extend([-ca.inf]*n_x); ubw.extend([ca.inf]*n_x)
    
    g.append(Xk - X_next)
    lbg.extend([0]*n_x); ubg.extend([0]*n_x)

# Terminal cost
J += Xk.T @ P @ Xk

qp_struct = {'f': J, 'x': ca.vertcat(*w), 'g': ca.vertcat(*g), 'p': x_init}

# Options for OSQP
opts = {'osqp': {'verbose': False}}

# CREATE SOLVER (once)
solver = ca.qpsol('solver', 'osqp', qp_struct, opts)

curr_x=np.array([[10],[0]])
solver(p=curr_x, lbx=lbw, ubx=ubw, lbg=lbg, ubg=ubg)

#u_control = sol['x'][n_x] # Take the first input after state x0


## 3rd attempt - Casadi Function usage for faster evaluation of constraints and costs

In [ ]:
import casadi as ca
import numpy as np

def cost_fun_build(X, U, X_all, U_all, C, P, R, N):
    f_cost = ca.Function('f_cost', [X, U],
                         [X.T @ C.T @ Q @ C @ X + U.T @ R @ U])
    stage_costs = f_cost.map(N)(X_all[:, :N], U_all)
    J = ca.sum2(stage_costs) + X_all[:, N].T @ P @ X_all[:, N]
    return J

def constraint_build(X, U, X_all, U_all, A, B, N, x_init):
    f_step = ca.Function('f_step', [X, U], [ A @ X + B @ U])
    X_next_sim = f_step.map(N)(X_all[:, :N], U_all)
    
    g_dyn = X_all[:, 1:] - X_next_sim
    g_init = X_all[:, 0] - x_init
    g = ca.vertcat(g_init, ca.reshape(g_dyn, -1, 1))
    return g


#Qc = C.T@Q@C

def set_controller(x_init, z, J, g, solver, options={}):
    qp = {'x': z, 'f': J, 'g': g, 'p': x_init}
    return ca.qpsol('qpsol', solver, qp, options)

def solve(qpsol, x0, lbw, ubw, lbg, ubg):
    return qpsol(p=x0, lbx=lbw, ubx=ubw, lbg=lbg, ubg=ubg)

In [ ]:
from pycvxset import Polytope
n_x,n_u, n_y, N = 2, 1, 1, 2

A = ca.DM([[1,1], [0, 1]])
B = ca.DM([[0], [1]])
C = ca.DM([[1, 0]])
Q, R, P = ca.DM(1), ca.DM(0.1), ca.DM(np.eye(2))

Qc = C.T@Q@C

_Q = ca.MX.sym('_Q',n_x,n_x)
_R = ca.MX.sym('_R',n_u,n_u)  
_x = ca.MX.sym('_x',n_x)
_u = ca.MX.sym('_u',n_u)  

quad_cost = ca.Function('quad_cost',[_x,_u,_Q,_R],[_x.T@_Q@_x + _u.T@_R@_u])  

X = ca.MX.sym('x', n_x)
U = ca.MX.sym('u', n_u)
X_all = ca.MX.sym('X_all', n_x, N+1)
U_all = ca.MX.sym('U_all', n_u, N)

x_a = ca.MX.sym('xa', n_x)
X_a = ca.MX.sym('Xa', n_x, N)


state_quad = ca.Function('state_quad', [X],
                             [X.T @ Q @ X])

#output_quad = ca.Function('output_quad', [X, Qc],
#                             [X.T @ Qc @ X])


#input_quad = ca.Function('input_quad', [U, R],
#                             [U.T @ R @ U])

x_init = ca.MX.sym('x_init', n_x)

J = cost_fun_build(X, U, X_all, U_all, C, P, R, N)
g = constraint_build(X, U, X_all, U_all, A, B, N, x_init)

z = ca.vertcat(ca.reshape(X_all, -1, 1), ca.reshape(U_all, -1, 1))

P = Polytope(A=np.array([[1, 0], [0, 1], [-1, 0], [0, -1]]), b=np.array([1, 1, 1, 1]))
F = (P.A/P.b[:, np.newaxis])[:, :n_x*(N+1)]
G = (P.A/P.b[:, np.newaxis])[:, n_x*(N+1):]
F = ca.MX(F)
G = ca.MX(G)

g = ca.vertcat(g, ca.reshape(F, -1, 1), ca.reshape(G, -1, 1))
#z_reshaped=ca.vertcat(ca.reshape(ca.vertcat(X_all[:,:N], U_all),-1,1),X_all[:,N])


qpsol = set_controller(x_init, z, J, g, 'osqp')
x0 = np.array([[10], [0]])

lbx = [-ca.inf]*(n_x*(N+1))
ubx = [ca.inf]*(n_x*(N+1))

lbu = [-1.0]*n_u*N
ubu = [1.0]*n_u*N

lbw = lbx + lbu
ubw = ubx + ubu

n_g = n_x*N + n_x
lbg = [0]*n_g
ubg = [0]*n_g

print(F.shape)

lbg = lbg + [-ca.inf]*(ca.reshape(F,-1, 1).size(1))
ubg = ubg + [1]*(ca.reshape(F,-1, 1).size(1))

sol = solve(qpsol, x0, lbw, ubw, lbg, ubg)

In [ ]:
u_star = sol['x'][(N+1)*n_x : (N+1)*n_x+n_u]
u_star,sol['x']


In [ ]:
import casadi as ca
import numpy as np

# --- System Setup ---
n_x, n_u, n_y = 2, 1, 1
N = 2 # Longer horizon for testing

# Matrices (use DM for numeric constants, MX for symbolic)
A = ca.DM([[1, 1], [0, 1]])
B = ca.DM([[0], [1]])
C = ca.DM([[1, 0]])
Q, R, P = 1, 0.1, np.eye(2)

# Support symbols (local to the function)
X = ca.MX.sym('x', n_x)
U = ca.MX.sym('u', n_u)

# Step function: computes the next state
# (x, u) -> (x_next)
f_step = ca.Function('f_step', [X, U], [A @ X + B @ U])

# Stage cost function: computes the cost for one time step
# (x, u) -> (cost)
f_cost = ca.Function('f_cost', [X, U], [X.T @ C.T @ Q @ C @ X + U.T @ R @ U])

# Decision variables (bulk)
U_all = ca.MX.sym('U', n_u, N)      # Matrix n_u x N (all inputs)
X_all = ca.MX.sym('X', n_x, N+1)    # Matrix n_x x (N+1) (all states)

# --- MAP for Cost ---
# Apply f_cost to each column of X_all and U_all simultaneously
stage_costs = f_cost.map(N)(X_all[:, :N], U_all)
J = ca.sum2(stage_costs) + X_all[:, N].T @ P @ X_all[:, N]

# --- MAP for Dynamics Constraints ---
# Apply f_step to all pairs (x_k, u_k) to obtain all (x_next_k)
X_next_sim = f_step.map(N)(X_all[:, :N], U_all)

# Equality constraints: x_{k+1} - f(x_k, u_k) = 0
g_dyn = X_all[:, 1:] - X_next_sim
# Initial state constraint: x_0 - x_init = 0
g_init = X_all[:, 0] - x_init

g = ca.vertcat(g_init, ca.reshape(g_dyn, -1, 1))


# Single decision vector
w = ca.vertcat(ca.reshape(X_all, -1, 1), ca.reshape(U_all, -1, 1))

qp_struct = {'f': J, 'x': w, 'g': g, 'p': x_init}
solver = ca.qpsol('solver', 'osqp', qp_struct)

# Bounds for X (n_x * (N+1) elements)
lbx = [-ca.inf] * (n_x * (N + 1))
ubx = [ca.inf] * (n_x * (N + 1))

# Bounds for U (n_u * N elements)
lbu = [-1.0] * (n_u * N)  # Example: saturation at -1
ubu = [1.0] * (n_u * N)   # Example: saturation at 1

# Combine variable bounds
lbw = lbx + lbu
ubw = ubx + ubu

# g_init has dimension n_x (x0 - x_init = 0)
# g_dyn has dimension n_x * N (dynamics x_{k+1} - Ax - Bu = 0)
n_constraints = n_x + (n_x * N)

lbg = [0.0] * n_constraints
ubg = [0.0] * n_constraints

# Example of a current initial state
curr_x = np.array([10.0, 0.0])

# Solve
solver(
    p=curr_x,      # Pass the initial state to the x_init parameter
    lbx=lbw,       # Bounds on variables (States and Inputs)
    ubx=ubw, 
    lbg=lbg,       # Bounds on constraints (Dynamics = 0)
    ubg=ubg
)


NameError: name 'x_init' is not defined

# Class Debug

In [1]:
from cMPC import CMPC
import numpy as np

import os
import mosek

# os.environ["MOSEKLM_LICENSE_FILE"] = "27007@192.168.100.118"
# mosek.Env().getversion()
# mosek.Env().putlicensepath("27007@192.168.100.118")
# mosek.Env().expirylicenses()

A = np.array([[1,1], [0, 1]])
B = np.array([[0], [1]])
C = np.array([[1, 0]])
Q, R = np.eye(2), np.array([[.1]])
opt = {
    'solver':'osqp',
    'verbose':True,
    'svd':False,
    'uBound':(np.array(-1),np.array(1)),
    'name':'test_mpc'
}

myMPC = CMPC({'A':A,'B':B,'C':C},Q,R,2,opt)
myMPC.initialize()
curr_x = np.array([10.0, 0.0])
myMPC.solve(curr_x)
myMPC.u_star



/Users/pascalbraun/Documents/Git/pycvxset/pycvxset/Polytope/__init__.py:398: UserWarning: Removed some rows in A that had all zeros | b that had np.inf!
  warnings.warn("Removed some rows in A that had all zeros | b that had np.inf!", UserWarning)


Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
Iteration: 7
Iteration: 8
Iteration: 9
Iteration: 10
Iteration: 11
Iteration: 12
Iteration: 13
Iteration: 14
Iteration: 15
Iteration: 16
Iteration: 17
Iteration: 18
Iteration: 19
Iteration: 20
Iteration: 21
Iteration: 22
Iteration: 23
Iteration: 24
Iteration: 25
Iteration: 26
Iteration: 27
Iteration: 28
Iteration: 29
Iteration: 30
Iteration: 31
Iteration: 32
Iteration: 33
Iteration: 34
Iteration: 35
Iteration: 36
Iteration: 37
Iteration: 38
Iteration: 39
Iteration: 40
Iteration: 41
Iteration: 42
Iteration: 43
Iteration: 44
Iteration: 45
Iteration: 46
Iteration: 47
Iteration: 48
Iteration: 49
Iteration: 50
Iteration: 51
Iteration: 52
Iteration: 53
Iteration: 54
Iteration: 55
Iteration: 56
Iteration: 57
Iteration: 58
Iteration: 59
Iteration: 60
Iteration: 61
Iteration: 62
Iteration: 63
Iteration: 64
Iteration: 65
Iteration: 66
Iteration: 67
Iteration: 68
Iteration: 69
Iteration: 70
Iteration: 71
It

DM(-1.01081)

In [1]:
from pycvxset import Polytope
import numpy as np

import os
import mosek

INF = 1e16

os.environ["MOSEKLM_LICENSE_FILE"] = "27007@192.168.100.118"
mosek.Env().getversion()
mosek.Env().putlicensepath("27007@192.168.100.118")
mosek.Env().expirylicenses()

A = np.vstack((np.eye(3),-np.eye(3)))
b = np.array([INF, INF, 1, INF, INF, 1])

P = Polytope(A=A,b=b)
P.minimize_H_rep()
P.A



V = np.array([[INF , -INF , 1],
              [INF , -INF , -1],
              [INF , INF , 1],
              [INF , INF , -1],
              [-INF , INF , 1],
              [-INF , INF , -1],
              [-INF , -INF , 1],
              [-INF , -INF , -1]])

P2=Polytope(V=V)
P2.A



S=P2.projection(2)
P3 = P2.projection(0)
P2.projection(2)
#S.intersection_with_halfspaces(P)

Error: rescode.err_license_cannot_connect(1021): MOSEK cannot connect to the license server.
